In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/katabatic

Mounted at /content/drive
/content/drive/MyDrive/katabatic1
 CONTRIBUTING.md		       main.py		        README.md
'C:\Users\Prabu\Downloads\Katabatic'   Makefile		        Results
 dev_deps.py			       medgan_adult.py	        runs
 discretized_data		       MODEL_CONTRIBUTIONS.md   sample_data
 encoded_data			       outputs		        scripts
 example.ipynb			       poetry.lock	        synthetic
 examples			       __pycache__	        utils.py
 katabatic			       pyproject.toml	        venv
 LICENSE			       raw_data


In [ ]:
!pip -q install --upgrade --force-reinstall \
  "numpy==2.0.2" \
  "pandas==2.2.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 34.3 MB/s eta 0:00:00


In [ ]:

import os, sys
os.kill(os.getpid(), 9)


In [ ]:
import numpy as np, pandas as pd
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


In [2]:
%cd /content/drive/MyDrive/katabatic1
!ls


/content/drive/MyDrive/katabatic1
 CONTRIBUTING.md		       main.py		        README.md
'C:\Users\Prabu\Downloads\Katabatic'   Makefile		        Results
 dev_deps.py			       medgan_adult.py	        runs
 discretized_data		       MODEL_CONTRIBUTIONS.md   sample_data
 encoded_data			       outputs		        scripts
 example.ipynb			       poetry.lock	        synthetic
 examples			       __pycache__	        utils.py
 katabatic			       pyproject.toml	        venv
 LICENSE			       raw_data


In [ ]:
!grep -R "def build_data_xt" -n /content/drive/MyDrive/katabatic1/katabatic/models/forestdiffusion || true
!grep -R "class IterForDMatrix" -n /content/drive/MyDrive/katabatic1/katabatic/models/forestdiffusion || true
!grep -R "def euler_solve" -n /content/drive/MyDrive/katabatic1/katabatic/models/forestdiffusion || true
!grep -R "def get_xt" -n /content/drive/MyDrive/katabatic1/katabatic/models/forestdiffusion || true


In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.forestdiffusion.adapter import ForestDiffusionAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "magic.csv"
SAMPLE_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR = ROOT / "synthetic" / "magic" / "forestdiffusion"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

fd_kwargs = dict(
    n_t=50,
    model="xgboost",
    n_estimators=100,
    max_depth=7,
    seed=666,
    gpu_hist=False,
)

print("Step 1: Train–Test Split")
t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: ForestDiffusionAdapter(**fd_kwargs)
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

print(f"Split completed in {time.time() - t0:.2f}s")

print("Step 2: Training ForestDiffusion")
t1 = time.time()

fd = ForestDiffusionAdapter(**fd_kwargs)
fd.train(output_dir=str(SAMPLE_DIR), label_col="class")

print(f"Training completed in {(time.time() - t1)/60:.2f} minutes")

print("Step 3: Generating synthetic data")
t2 = time.time()

n_samples = len(pd.read_csv(SAMPLE_DIR / "x_train.csv"))
x_synth, y_synth = fd.sample(n_samples=n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({"class": y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print(f"Synthetic data saved to: {SYNTH_DIR}")
print(f"Generation time: {(time.time() - t2)/60:.2f} minutes")

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Feature names aligned for TSTR")

print("Step 4: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()
print(results)

print("PIPELINE COMPLETE")
print(f"Total runtime: {(time.time() - t0)/60:.2f} minutes")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

⏳ Step 1: Train–Test Split
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
✅ Split completed in 26.91s

⏳ Step 2: Training ForestDiffusion
✅ Training completed in 0.45 minutes

⏳ Step 3: Generating synthetic data
✅ Synthetic data saved to: /content/drive/MyDrive/katabatic1/synthetic/magic/forestdiffusion
⏱️ Generation time: 0.14 minutes
✅ Feature names aligned for TSTR

⏳ Step 4: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/magic/forestdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6483
F1 Score: 0.5099
AUC: 0.5520

MLP:
Accuracy: 0.6640
F1 Score: 0.5745
AUC: 0.4771

RF:
Accuracy: 0.6272
F1 Score: 0.5213
AUC: 0.4914

XGBoost:
Accuracy: 0.5284
F1 Score: 0.5339
AUC: 0.4914

🎉 PIPELINE COMPLETE
⏱️ Total runtime: 1.32 minutes


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.forestdiffusion.adapter import ForestDiffusionAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "car.csv"
SAMPLE_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR = ROOT / "synthetic" / "car" / "forestdiffusion"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

fd_kwargs = dict(
    n_t=50,
    model="xgboost",
    n_estimators=100,
    max_depth=7,
    seed=666,
    gpu_hist=False,
)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: ForestDiffusionAdapter(**fd_kwargs)
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="6"
)
from sklearn.preprocessing import OrdinalEncoder

# Load raw feature CSVs
x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df  = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

# Fit encoder ONLY on real training data
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

# Transform all feature sets
x_train_enc = encoder.transform(x_train_df.astype(str))
x_test_enc  = encoder.transform(x_test_df.astype(str))
x_synth_enc = encoder.transform(x_synth_df.astype(str))

# Save back (overwrite for TSTR)
pd.DataFrame(x_train_enc).to_csv(SAMPLE_DIR / "x_train.csv", index=False)
pd.DataFrame(x_test_enc).to_csv(SAMPLE_DIR / "x_test.csv", index=False)
pd.DataFrame(x_synth_enc).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

print("All features encoded numerically for TSTR")


fd = ForestDiffusionAdapter(**fd_kwargs)
fd.train(output_dir=str(SAMPLE_DIR), label_col="6")

n_samples = len(pd.read_csv(SAMPLE_DIR / "x_train.csv"))
x_synth, y_synth = fd.sample(n_samples=n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({"6": y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()
print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
All features encoded numerically for TSTR


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [04:59:41] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/forestdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.5636
F1 Score: 0.5042

MLP:
Accuracy: 0.5058
F1 Score: 0.4750

RF:
Accuracy: 0.5173
F1 Score: 0.4806

XGBoost:
Accuracy: 0.4942
F1 Score: 0.4672
{'LR': {'Accuracy': 0.5635838150289018, 'F1 Score': 0.5042043742587586}, 'MLP': {'Accuracy': 0.5057803468208093, 'F1 Score': 0.4750497024925534}, 'RF': {'Accuracy': 0.5173410404624278, 'F1 Score': 0.48058982970445285}, 'XGBoost': {'Accuracy': 0.49421965317919075, 'F1 Score': 0.46723292144649003}}
Total runtime (minutes): 0.37


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import numpy as np
import pandas as pd
from pathlib import Path

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.forestdiffusion.adapter import ForestDiffusionAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation


ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "shuttle.csv"
SAMPLE_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR  = ROOT / "synthetic" / "shuttle" / "forestdiffusion"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

fd_kwargs = dict(
    n_t=50,
    model="xgboost",
    n_estimators=100,
    max_depth=7,
    seed=666,
    gpu_hist=False,
)

t0 = time.time()

print("Step 1: Train–Test Split")

pipeline = TrainTestSplitPipeline(
    model=lambda: ForestDiffusionAdapter(**fd_kwargs)
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"
)

print("Step 2: Training ForestDiffusion")

fd = ForestDiffusionAdapter(**fd_kwargs)
fd.train(output_dir=str(SAMPLE_DIR), label_col="class")

print("Step 3: Generating synthetic data")

n_samples = len(pd.read_csv(SAMPLE_DIR / "x_train.csv"))
x_synth, y_synth = fd.sample(n_samples=n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({"class": y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Aligning feature names")

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")
x_test.columns = x_synth_df.columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Fixing synthetic label encoding")

y_real = pd.read_csv(SAMPLE_DIR / "y_train.csv")["class"].values
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")["class"].values

unique_real = sorted(np.unique(y_real))
unique_synth = sorted(np.unique(y_synth))

label_map = {s: r for s, r in zip(unique_synth, unique_real)}
y_synth_fixed = pd.Series(y_synth).map(label_map).values

pd.DataFrame({"class": y_synth_fixed}).to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Step 4: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Train–Test Split
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
Step 2: Training ForestDiffusion
Step 3: Generating synthetic data
Aligning feature names
Fixing synthetic label encoding
Step 4: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [05:31:43] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/forestdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6109
F1 Score: 0.5968

MLP:
Accuracy: 0.7686
F1 Score: 0.6861

RF:
Accuracy: 0.7740
F1 Score: 0.6863

XGBoost:
Accuracy: 0.7742
F1 Score: 0.6876
{'LR': {'Accuracy': 0.610948275862069, 'F1 Score': 0.5968139652231604}, 'MLP': {'Accuracy': 0.7686206896551724, 'F1 Score': 0.6861367695668469}, 'RF': {'Accuracy': 0.7739655172413793, 'F1 Score': 0.6862715056107427}, 'XGBoost': {'Accuracy': 0.7742241379310345, 'F1 Score': 0.6876377462754254}}
Total runtime (minutes): 4.15


In [7]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.forestdiffusion.adapter import ForestDiffusionAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "forestdiffusion"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

fd_kwargs = dict(
    n_t=50,
    model="xgboost",
    n_estimators=100,
    max_depth=7,
    seed=666,
    gpu_hist=False,
)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: ForestDiffusionAdapter(**fd_kwargs)
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"
)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df  = pd.read_csv(SAMPLE_DIR / "x_test.csv")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

x_train_enc = encoder.transform(x_train_df.astype(str))
x_test_enc  = encoder.transform(x_test_df.astype(str))

pd.DataFrame(x_train_enc).to_csv(SAMPLE_DIR / "x_train.csv", index=False)
pd.DataFrame(x_test_enc).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

fd = ForestDiffusionAdapter(**fd_kwargs)
fd.train(output_dir=str(SAMPLE_DIR), label_col="class")

n_samples = len(pd.read_csv(SAMPLE_DIR / "x_train.csv"))
x_synth, y_synth = fd.sample(n_samples=n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({"income": y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")
y_test.columns = ["income"]
y_test.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/forestdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7593
F1 Score: 0.6553
AUC: 0.6151

MLP:
Accuracy: 0.7582
F1 Score: 0.6557
AUC: 0.5210

RF:
Accuracy: 0.7591
F1 Score: 0.6553
AUC: 0.5420

XGBoost:
Accuracy: 0.5959
F1 Score: 0.6245
AUC: 0.6135
{'LR': {'Accuracy': 0.7592507293106096, 'F1 Score': 0.6553490760064522, 'AUC': np.float64(0.6150940962836096)}, 'MLP': {'Accuracy': 0.7581759557807461, 'F1 Score': 0.6556803434639428, 'AUC': np.float64(0.5209647964342461)}, 'RF': {'Accuracy': 0.7590971902349147, 'F1 Score': 0.6552737375773158, 'AUC': np.float64(0.5420309630424465)}, 'XGBoost': {'Accuracy': 0.5958851527713803, 'F1 Score': 0.6245100562824984, 'AUC': np.float64(0.6135284042838571)}}
Total runtime (minutes): 4.16


In [8]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.forestdiffusion.adapter import ForestDiffusionAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "nursery.csv"
SAMPLE_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR  = ROOT / "synthetic" / "nursery" / "forestdiffusion"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

fd_kwargs = dict(
    n_t=50,
    model="xgboost",
    n_estimators=100,
    max_depth=7,
    seed=666,
    gpu_hist=False,
)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: ForestDiffusionAdapter(**fd_kwargs)
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

fd = ForestDiffusionAdapter(**fd_kwargs)

# After split, y_train.csv is its own file (usually first column or "target")
fd.train(output_dir=str(SAMPLE_DIR), label_col=None)

n_samples = len(pd.read_csv(SAMPLE_DIR / "x_train.csv"))
x_synth, y_synth = fd.sample(n_samples=n_samples)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
pd.DataFrame(x_synth, columns=x_train_df.columns).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({"target": y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df  = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str)),
    columns=x_train_df.columns
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str)),
    columns=x_test_df.columns
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

pd.DataFrame(
    encoder.transform(x_synth_df.astype(str)),
    columns=x_synth_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [06:05:51] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/forestdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.3333
F1 Score: 0.1667

MLP:
Accuracy: 0.3291
F1 Score: 0.1630

RF:
Accuracy: 0.3333
F1 Score: 0.1667

XGBoost:
Accuracy: 0.3333
F1 Score: 0.1667
{'LR': {'Accuracy': 0.3333333333333333, 'F1 Score': 0.16666666666666666}, 'MLP': {'Accuracy': 0.3290895061728395, 'F1 Score': 0.16296856241824795}, 'RF': {'Accuracy': 0.3333333333333333, 'F1 Score': 0.16666666666666666}, 'XGBoost': {'Accuracy': 0.3333333333333333, 'F1 Score': 0.16666666666666666}}
Total runtime (minutes): 0.85
